# ¿Qué hogares presentan inseguridad alimentaria?
**Taller práctico de 90 minutos — ENSANUT 2025**

Vamos a describir los hogares y entrenar un árbol de decisión. Al final quedan otros algoritmos para practicar en casa.

## 1. Cargar los datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

URL_DATOS = "https://raw.githubusercontent.com/jdelacruzm/AnalisisDatos/main/ensanut2025_hogares.csv"
hogares = pd.read_csv(URL_DATOS)

print("Hogares encuestados:", len(hogares))
display(hogares.head())

## 2. Explorar la inseguridad alimentaria
El factor de expansión se utiliza para estimar porcentajes de hogares representados por la muestra.

In [ ]:
niveles = ["Seguridad", "Leve", "Moderada", "Severa"]
porcentajes = hogares.groupby("inseguridad_alimentaria")["factor_expansion"].sum()
porcentajes = 100 * porcentajes / porcentajes.sum()
display(porcentajes.reindex(niveles).round(1))

In [ ]:
def porcentaje_ponderado(datos):
    afectados = datos["inseguridad_moderada_o_severa"] == "Sí"
    return 100 * datos.loc[afectados, "factor_expansion"].sum() / datos["factor_expansion"].sum()

print(f"Inseguridad moderada o severa: {porcentaje_ponderado(hogares):.1f}%")

### Comparar grupos de hogares

In [ ]:
factores = ["ambito", "jefatura", "hay_menores", "nivel_socioeconomico",
            "recibe_programa_social", "escolaridad_jefe"]

tablas = {}
for factor in factores:
    tablas[factor] = hogares.groupby(factor).apply(
        lambda grupo: porcentaje_ponderado(grupo), include_groups=False
    ).round(1)
    print("\n", factor)
    display(tablas[factor])

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 4))
for eje, factor in zip(ejes, ["nivel_socioeconomico", "ambito", "jefatura"]):
    tablas[factor].sort_values().plot.barh(ax=eje, title=factor)
    eje.set_xlabel("Porcentaje")
plt.tight_layout()
plt.show()

## 3. Árbol de decisión
Primero definimos las características (`X`) y el resultado que queremos clasificar (`y`).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

orden_escolaridad = ["sin escolaridad", "primaria o menos", "secundaria",
                    "preparatoria", "licenciatura o más"]

X = pd.DataFrame({
    "Hogar rural": (hogares["ambito"] == "Rural").astype(int),
    "Jefatura femenina": (hogares["jefatura"] == "Femenina").astype(int),
    "Integrantes": hogares["integrantes"],
    "Hay menores": (hogares["hay_menores"] == "Sí").astype(int),
    "Nivel socioeconómico bajo": hogares["nivel_socioeconomico"].str[0].astype(int).le(2).astype(int),
    "Recibe programa social": (hogares["recibe_programa_social"] == "Sí").astype(int),
    "Escolaridad del jefe": hogares["escolaridad_jefe"].str.lower().map(
        {valor: i for i, valor in enumerate(orden_escolaridad)})
})
y = (hogares["inseguridad_moderada_o_severa"] == "Sí").astype(int)
pesos = hogares["factor_expansion"]

### Separar datos para entrenamiento y prueba

In [ ]:
X_entrena, X_prueba, y_entrena, y_prueba, w_entrena, w_prueba = train_test_split(
    X, y, pesos, test_size=0.30, random_state=2025, stratify=y
)
print("Entrenamiento:", len(X_entrena), "| Prueba:", len(X_prueba))

### Entrenar y dibujar el árbol

In [ ]:
arbol = DecisionTreeClassifier(max_depth=3, min_samples_leaf=120,
                               class_weight="balanced", random_state=2025)
arbol.fit(X_entrena, y_entrena, sample_weight=w_entrena)

In [ ]:
plt.figure(figsize=(20, 9))
plot_tree(arbol, feature_names=X.columns, class_names=["No", "Sí"],
          impurity=False, proportion=True, rounded=True, fontsize=10)
plt.show()

### Perfiles que encuentra el árbol
Cada hoja representa un grupo de hogares. Observa sus reglas en el dibujo y compara qué porcentaje presenta inseguridad alimentaria.

In [ ]:
perfiles = pd.DataFrame({
    "Hoja": arbol.apply(X_prueba),
    "Inseguridad": y_prueba.to_numpy(),
    "Peso": w_prueba.to_numpy()
})

resumen = perfiles.groupby("Hoja").apply(
    lambda grupo: pd.Series({
        "Hogares de prueba": len(grupo),
        "Inseguridad (%)": 100 * grupo.loc[grupo["Inseguridad"] == 1, "Peso"].sum() / grupo["Peso"].sum()
    }), include_groups=False
)
display(resumen.sort_values("Inseguridad (%)", ascending=False).round(1))

## 4. Evaluar el modelo
La matriz de confusión muestra los aciertos y errores del árbol con hogares que no utilizó para aprender.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, recall_score, precision_score, f1_score, roc_auc_score

predicciones = arbol.predict(X_prueba)
ConfusionMatrixDisplay.from_predictions(y_prueba, predicciones, display_labels=["No", "Sí"])
plt.show()

In [ ]:
metricas = pd.Series({
    "Exactitud": accuracy_score(y_prueba, predicciones),
    "Sensibilidad": recall_score(y_prueba, predicciones),
    "Precisión": precision_score(y_prueba, predicciones),
    "F1": f1_score(y_prueba, predicciones),
    "AUC": roc_auc_score(y_prueba, arbol.predict_proba(X_prueba)[:, 1])
})
display(metricas.round(2))

### ¿Qué características utiliza más el árbol?
La importancia de una variable no demuestra causalidad.

In [ ]:
importancia = pd.Series(arbol.feature_importances_, index=X.columns)
importancia.sort_values().plot.barh(title="Importancia de las variables")
plt.show()

---
## Para practicar en casa: otros algoritmos
Usa exactamente los mismos datos de entrenamiento y prueba. Puedes comparar regresión logística, bosque aleatorio, gradient boosting y SVM con el árbol de decisión.

**Nota:** los modelos se comparan sobre el conjunto de prueba; sus resultados no deben interpretarse como relaciones causales.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

modelos = {
    "Árbol": arbol,
    "Regresión logística": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=50, class_weight="balanced", random_state=2025),
    "Gradient Boosting": HistGradientBoostingClassifier(max_depth=3, class_weight="balanced", random_state=2025),
    "SVM": make_pipeline(StandardScaler(), SVC(class_weight="balanced", random_state=2025))
}

In [ ]:
resultados = []
for nombre, modelo in modelos.items():
    if nombre != "Árbol":
        if nombre in ["Regresión logística", "SVM"]:
            modelo.fit(X_entrena, y_entrena, **{modelo.steps[-1][0] + "__sample_weight": w_entrena})
        else:
            modelo.fit(X_entrena, y_entrena, sample_weight=w_entrena)
    pred = modelo.predict(X_prueba)
    resultados.append({
        "Modelo": nombre,
        "Exactitud": accuracy_score(y_prueba, pred),
        "Sensibilidad": recall_score(y_prueba, pred),
        "F1": f1_score(y_prueba, pred)
    })

display(pd.DataFrame(resultados).round(2))

## Discusión final
- ¿Qué grupos presentan mayor inseguridad alimentaria?
- ¿Qué características aparecen en las primeras divisiones del árbol?
- ¿Qué significa que el modelo no detecte a un hogar afectado?
- ¿Qué resultados sería necesario contrastar con la teoría y el trabajo de campo?

Puedes pedir a una IA que **interprete los resultados**, pero verifica sus afirmaciones con el análisis y la documentación de la encuesta.